In [ ]:
from pathlib import Path
import os
import pandas as pd

# Find the project root reliably, even if the notebook runs from a different folder

def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists() and (candidate / "code").exists():
            return candidate
    return current

root = find_repo_root(Path.cwd())

# Try the exact files you mentioned, using the repo root so paths work correctly
possible_files = [
    root / "data" / "FixForesight-cleaneddataset.csv",
    root / "data" / "FixForesightdataset.csv",
    root / "data" / "engineered_ai4i.csv",
    Path("data/FixForesight-cleaneddataset.csv"),
    Path("data/FixForesightdataset.csv"),
    Path("data/engineered_ai4i.csv")
]

RAW_FILE = None
for path in possible_files:
    if path.exists():
        RAW_FILE = path
        break

if RAW_FILE is None:
    raise FileNotFoundError(
        "Dataset not found. Checked these locations:\n" +
        "\n".join(str(p) for p in possible_files)
    )

OUTPUT_FILE = root / "data" / "ai4i2020_cleaned.csv"

print(f"Using dataset: {RAW_FILE}")
print(f"Project root: {root}")

# Read CSV or Excel depending on the file type
try:
    if RAW_FILE.suffix.lower() in {".xlsx", ".xls"}:
        df = pd.read_excel(RAW_FILE)
    else:
        # Some files may look like CSV but actually contain Excel content
        try:
            df = pd.read_csv(RAW_FILE)
        except Exception:
            df = pd.read_excel(RAW_FILE)
except Exception as e:
    raise RuntimeError(f"Could not read file {RAW_FILE}: {e}") from e

print("Raw shape:", df.shape)
print("Missing values before cleaning:\n", df.isna().sum())
print("Duplicate rows before cleaning:", df.duplicated().sum())

# Clean column names and text values
df.columns = df.columns.str.strip()

string_cols = df.select_dtypes(include=["object", "string"]).columns
for col in string_cols:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace({"nan": None, "None": None})

# Convert numeric columns safely
for col in df.columns:
    if col not in string_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Remove duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

print("Missing values after cleaning:\n", df.isna().sum())
print("Duplicate rows after cleaning:", df.duplicated().sum())
print("Final shape:", df.shape)

# Save the cleaned output
df.to_csv(OUTPUT_FILE, index=False)
print("Saved cleaned dataset to:", OUTPUT_FILE)

Using dataset: C:\Users\Anansh\Desktop\CODING PROGRAMs\Industrial-AI-Project\data\FixForesight-cleaneddataset.csv
Project root: C:\Users\Anansh\Desktop\CODING PROGRAMs\Industrial-AI-Project
Raw shape: (10000, 14)
Missing values before cleaning:
 UDI                        0
Product ID                 0
Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Machine failure            0
TWF                        0
HDF                        0
PWF                        0
OSF                        0
RNF                        0
dtype: int64
Duplicate rows before cleaning: 0
Missing values after cleaning:
 UDI                        0
Product ID                 0
Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Machine failure            0
TWF     

: 